# Dimensionality reduction notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.ipynb -> dimensionality_reduction.ipynb -> gmm_clustering.ipynb`

In [ ]:
from __future__ import annotations

import typing
from pathlib import Path

import numpy as np
import pandas as pd
import plotnine as p9
from plotnine.composition import plot_spacer
from sklearn import set_config
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from utils.dimensionality_reduction_utils import BlockPCA
from utils.misc_utils import turn_cats_to_continuous
from utils.plotting_utils import make_plot_2d_data, make_plot_scree
from utils.saving_utils import load_data, save_data, save_plot

pd.options.mode.copy_on_write = True

# Important for pipeline to work, since BlockPCA maps pd.Dataframes
set_config(transform_output="pandas")

In [ ]:
# Read in data
data_discov_imputed_path = Path("outputs/imputed/data_discov_imputed.pkl")
data_valid_imputed_path = Path("outputs/imputed/data_valid_imputed.pkl")
data_tf_imputed_path = Path("outputs/imputed/data_tf_imputed.pkl")

vars_info_path = Path("data/preprocessed/vars_info_reduced.pkl")

data_discov_imputed: pd.DataFrame = load_data(data_discov_imputed_path)
data_valid_imputed: pd.DataFrame = load_data(data_valid_imputed_path)
data_tf_imputed: pd.DataFrame = load_data(data_tf_imputed_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# Set outputs folder
outputs_folder = "outputs/dim_red"

# Set number of principal components by group
n_components_per_group = 7  # dict also possible: {"alcohol": 4, "nicotine": 3, "other_drugs": 4, "medication": 3}

In [ ]:
# Get groups and find binary and ordinal variables which need to be converted to continuous ones
groups = vars_info["group"].cat.categories
var_groups = {
    group: vars_info.loc[vars_info["group"] == group, "name"].to_list()  # type: ignore
    for group in groups
}
treat_cat_as_continuous: list[str] = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"].tolist()

In [ ]:
# Set up models
pca_discov = make_pipeline(
    StandardScaler(),
    BlockPCA(var_groups=var_groups, n_components_per_group=n_components_per_group, rng=np.random.default_rng(3295)),
)
pca_valid = make_pipeline(
    StandardScaler(),
    BlockPCA(var_groups=var_groups, n_components_per_group=n_components_per_group, rng=np.random.default_rng(98634)),
)

# Fit models
pca_discov.fit(turn_cats_to_continuous(data_discov_imputed, treat_cat_as_continuous))
pca_valid.fit(turn_cats_to_continuous(data_valid_imputed, treat_cat_as_continuous))
None

In [ ]:
# Apply models (cast to pd.Dataframe because of set_config(transform_output="pandas") at the start of the file)
data_discov_pca = typing.cast(
    pd.DataFrame, pca_discov.transform(turn_cats_to_continuous(data_discov_imputed, treat_cat_as_continuous))
)
data_valid_pca = typing.cast(
    pd.DataFrame, pca_valid.transform(turn_cats_to_continuous(data_valid_imputed, treat_cat_as_continuous))
)
data_tf_pca = typing.cast(
    pd.DataFrame, pca_discov.transform(turn_cats_to_continuous(data_tf_imputed, treat_cat_as_continuous))
)

In [ ]:
# Create plots of principal components
for group in groups:
    # Plot the first two principal components of one group
    data_discov_plot = data_discov_pca.loc[:, data_discov_pca.columns.str.startswith(group)]
    data_valid_plot = data_valid_pca.loc[:, data_valid_pca.columns.str.startswith(group)]
    data_tf_plot = data_tf_pca.loc[:, data_tf_pca.columns.str.startswith(group)]

    plot_discov_pca = make_plot_2d_data(data_discov_plot) + p9.theme(figure_size=(9, 6))
    plot_valid_pca = make_plot_2d_data(data_valid_plot) + p9.theme(figure_size=(9, 6))
    plot_tf_pca = make_plot_2d_data(data_tf_plot) + p9.theme(figure_size=(9, 6))

    # save_plot(plot_discov_pca, Path(outputs_folder, f"plot_discov_pca_{group}.png"), dpi=150)
    # save_plot(plot_valid_pca, Path(outputs_folder, f"plot_valid_pca_{group}.png"), dpi=150)
    # save_plot(plot_tf_pca, Path(outputs_folder, f"plot_tf_pca_{group}.png"), dpi=150)

    # Combine the three principal component plots
    plot_discov_pca += p9.labs(title=f"Discovery data {group} PCA")
    plot_valid_pca += p9.labs(title=f"Validation data {group} PCA")
    plot_tf_pca += p9.labs(title=f"Transferred data {group} PCA")
    plot_all_pca = plot_discov_pca + plot_spacer() + plot_valid_pca + plot_tf_pca

    plot_all_pca.show()

    save_plot(plot_all_pca, Path(outputs_folder, f"plot_all_pca_{group}.png"), dpi=150)

In [ ]:
# Create scree plots
for group in groups:
    # Create scree plots
    plot_scree_discov = make_plot_scree(pca_discov["blockpca"].models[group])  # type: ignore
    plot_scree_valid = make_plot_scree(pca_valid["blockpca"].models[group])  # type: ignore

    # save_plot(plot_scree_discov, Path(outputs_folder, f"plot_scree_discov_{group}.png"), dpi=150)
    # save_plot(plot_scree_valid, Path(outputs_folder, f"plot_scree_valid_{group}.png"), dpi=150)

    # Combine the two scree plots
    plot_scree_discov += p9.labs(title=f"Discovery {group} PCA scree plot")
    plot_scree_valid += p9.labs(title=f"Validation {group} PCA scree plot")
    plot_scree_all = plot_scree_discov + plot_scree_valid

    plot_scree_all.show()

    save_plot(plot_scree_all, Path(outputs_folder, f"plot_scree_all_{group}.png"), dpi=150)

In [ ]:
# Save models
save_data(pca_discov, save_path=Path(outputs_folder, "pca_discov.pkl"))
save_data(pca_valid, save_path=Path(outputs_folder, "pca_valid.pkl"))

# Save transformed data
save_data(data_discov_pca, Path(outputs_folder, "data_discov_pca.pkl"))
save_data(data_valid_pca, Path(outputs_folder, "data_valid_pca.pkl"))
save_data(data_tf_pca, Path(outputs_folder, "data_tf_pca.pkl"))